# Step 1 — Direct Spark submission: guided activities

Use this notebook as a **Markdown-only learning guide**. All analysis code is
provided in `01_spark_basics.py`; `spark_direct_submit.py` submits it to your assigned
remote client. Do not execute the displayed Python snippets in a notebook kernel.

For each activity: **read → predict → copy the command to the terminal → inspect
the output → record and explain your findings → continue**. Double-click a
"Your answers" cell to edit it, then render it and save your notebook.

## Before starting

Wait for organiser permission and confirmation of your assigned workspace.
**Participant execution remains unvalidated.** File publication and offline
checks do not establish readiness. `STEP1-DIRECT-SUBMISSION.md`
provides the operational access and reporting instructions. This notebook
provides the learning sequence; do not combine old and new copies of the files.

Use only the prepared environment. Do not install Java, PySpark or packages,
run the analysis file directly in JupyterLab, guess settings, weaken host-key
checks, or start/stop shared containers. **One active submission per group: take turns on your assigned client.** There is no launcher concurrency lock.

## Learning objectives

- Explain what the automated preparation supplies.
- Distinguish the transport, driver, master, worker and executors.
- Explain CSV loading, schemas and inferred types.
- Distinguish column selection from row filtering.
- Identify transformations and actions, and interpret grouped counts.
- Distinguish a remote bounded preview from its local download.
- Check completion and understand the limits of cleanup evidence.

## Understand the process that automation has prepared

You do not repeat the infrastructure setup during this workshop. The organiser
has prepared an assigned client and a VM-local Spark runtime, made the supplied
dataset accessible, configured Spark connection settings, and staged protected
group access credentials. These steps make the exercise reproducible; they do
not remove the need to understand its architecture.

Container setup concepts include **services** (components), **images** (packaged
software), **environment** (process settings), **ports** (network endpoints), and
**mounted storage** (data made accessible inside a container). These are concepts
to understand here, not instructions to change the deployment.

### Where execution happens

1. Your **JupyterLab terminal** runs the launcher, using SSH/SFTP transport.
2. The launcher uploads the analysis file to the assigned **remote client**.
3. Real `spark-submit` launches the Python application's **driver** in that client
   using client deployment mode. The driver plans work and coordinates results.
4. The **master** coordinates resource allocation. The assigned VM's **worker**
   launches **executors**, which execute tasks on data partitions.
5. The driver stops its Spark session when the activity finishes. Shared runtime
   containers remain running. Only an exporting activity triggers preview download.

The notebook and JupyterLab kernel do not host this Spark driver. Executors must
connect back to the driver; reaching the master alone is insufficient. This pilot
uses **one Spark worker within each assigned VM**, not a cluster spanning all
workshop VMs. A small CSV and grouped counts do not demonstrate scaling speedup.

The launcher uses the organiser-pinned host identity and your group's protected
key under `~/.ssh/spark-direct/`. The client supplies the master URL, distributed
data path and driver address. Participants do not choose or export those settings.
The existing one-core limit, 512 MB driver/executor memory limits and timeouts
are unchanged.

### Reflect before submitting

- Which process is the driver, and where does it run?
- How does a worker differ from an executor?
- Why are dataset access and driver reachability both necessary?
- Which setup work has automation removed from your participant tasks?

### Your answers

**Driver location and worker/executor distinction**

_Write your answer here._

**Dataset access and driver reachability**

_Write your answer here._

**What automation prepared and why**

_Write your answer here._

## Locate your files and check the submission path

In your assigned workspace, locate this notebook, `spark_direct_submit.py`,
`01_spark_basics.py` and `data/Bank_Personal_Loan_Modelling.csv`. Do not edit the
supplied CSV. Use the README and dataset documentation for provenance, permission
limitations, units and category definitions. The existing dataset review reports
negative `Experience` values; do not silently correct them or infer field meanings.

Open a **JupyterLab terminal** and copy:

```bash
cd ~/Big-Data/Spark
```

Before the test, predict: what will a successful SparkPi submission tell you,
and what will it leave untested?

After organiser permission, copy:

```bash
python3 spark_direct_submit.py test
```

Inspect the printed remote command: locate `spark-submit`, `--master`,
`--deploy-mode client`, the resource limits and `timeout 120`. Identify what
these options control. Record the SparkPi result and `Remote exit status:`.

**Continue only after successful test completion and organiser permission.**
SparkPi success does not establish bank-data access, preview retrieval, isolation,
or complete cleanup. After an error or interruption, stop and report it.

### Your answers

**Prediction: what the test establishes and leaves untested**

_Write your answer here._

**Observed result, exit status and submission options**

_Write your answer here._

## How the numbered activities work

| Activity | Analysis task | Downloads a preview? |
|---|---|---|
| 1 | Load CSV, inspect schema, count input records | No |
| 2 | Select columns | No |
| 3 | Filter rows | No |
| 4 | Count Personal Loan categories | No |
| 5 | Export a bounded filtered preview | Yes |

Activity numbers identify learning tasks, **not physical notebook cell positions**.
Adding explanation or answer cells does not change the commands.

Each command is a **separate Spark application**. It performs a bounded local
CSV preview in the remote client, reads prepared settings, starts its own Spark
session, loads the DataFrame it needs, performs its selected analysis, and attempts
` spark.stop()` before returning. DataFrames do not persist between commands.
Activity 5 recreates its filter; it does not reuse activity 3's live variables.
Repeated startup and data loading are part of this submission model.

Only the selected analysis activity runs, plus its necessary preparation and
cleanup. Loading with schema inference can itself require processing; it is not
correct to assume that all input preparation is free or deferred.

For each activity, inspect `Spark session stopped.`,
`Analysis completed successfully.` and `Remote exit status: 0`. These are normal
completion messages, not a guarantee after an interruption. Activities 1–4 should
**not** print `Downloaded preview:`. Answer the reflection questions before
continuing. If completion is uncertain, stop and ask the organiser before retrying.

## Activity 1 — Load the CSV and inspect its schema

**Purpose:** understand how a CSV becomes a Spark DataFrame with named, typed
columns. The remote Python CSV preview is ordinary file reading; the Spark read
uses the prepared distributed path. These paths can differ while referring to
the same supplied snapshot. A local preview alone does not prove executor access.

### Predict

- Which types do you expect for `Age`, `Income` and `Family`?
- Does a CSV declare a schema, or must Spark determine the types?
- Why is checking a few displayed records different from counting all records?

### Read the supplied code — do not execute it locally

```python
def load_dataframe(spark, settings: dict):
    """Read the prepared distributed CSV with a header and inferred types."""
    return (
        spark.read.format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load(settings["SPARK_DATA_PATH"])
    )

def inspect_schema(df) -> None:
    """Activity 1: inspect types, display five records and count all records."""
    df.printSchema()
    df.show(5)
    print("Number of records:", df.count())
```

### Copy to the terminal

```bash
python3 spark_direct_submit.py analyse --activity 1
```

### Observe and reflect

Record the Spark version, master URL, inferred types, full input count and
completion messages. Distinguish the ordinary local CSV preview from the Spark
DataFrame display. Record only relevant output, not all Spark logs.

- What do `header` and `inferSchema` do?
- Which fields are measurements, categories or counts? Consult the documentation.
- Why does showing five records not establish the size of the input?
- What would you investigate if a supposedly numeric field were inferred as text?

### Your answers

**Predictions before execution**

_Write your answer here._

**Observed types, input count, version/master and completion**

_Write your answer here._

**Explanation of loading, field meanings and schema checks**

_Write your answer here._

## Activity 2 — Select columns

**Purpose:** distinguish changing the columns from changing the records.
`select` is a transformation: it builds a plan. `show` and `count` request results
and are actions. Spark defers much of its execution until actions request results;
this is called lazy evaluation.

### Predict

Which columns will remain? Will the number of records differ from activity 1?

### Read the supplied code — do not execute it locally

```python
def select_columns(df) -> None:
    """Activity 2: select three columns without filtering any records."""
    selected_df = df.select("Age", "Income", "Family")
    selected_df.show(10)
    print("Selected columns:", ", ".join(selected_df.columns))
    print("Records after column selection:", selected_df.count())
```

The activity loads a fresh DataFrame using the same supplied loading function.

### Copy to the terminal

```bash
python3 spark_direct_submit.py analyse --activity 2
```

### Observe and reflect

Record the column names, record count and completion messages.

- Did your prediction match? Explain any difference.
- Why can the display contain ten records while the selected DataFrame contains more?
- Which lines are transformations and which lines are actions?
- Does this command reuse the DataFrame from the previous application?

### Your answers

**Prediction before execution**

_Write your answer here._

**Observed columns, count and completion**

_Write your answer here._

**Explanation of selection, actions and independent submissions**

_Write your answer here._

## Activity 3 — Filter rows using Age > 30

**Purpose:** understand a strict condition and distinguish selection from filtering.
The activity selects three columns and then retains only matching records.

### Predict

Will a record with age exactly 30 remain? Will the matching count be greater than,
equal to or smaller than the full input count? Explain your expectation.

### Read the supplied code — do not execute it locally

```python
def filtered_dataframe(df):
    """Build the strict Age > 30 filter used by activities 3 and 5."""
    return df.select("Age", "Income", "Family").filter(df.Age > 30)

def inspect_filter(result_df) -> None:
    """Activity 3: display ten filtered records and count all matches."""
    result_df.show(10)
    print("Matching records:", result_df.count())
```

### Copy to the terminal

```bash
python3 spark_direct_submit.py analyse --activity 3
```

### Observe and reflect

Record the matching count, columns, a small output excerpt and completion messages.

- Compare the matching count with the input count from activity 1.
- Why do records with age exactly 30 not match?
- Which operations build a plan, and which request execution?
- Does the displayed preview contain every matching record?
- No explicit row ordering is applied. Should reruns be expected to display exactly
  the same first ten rows? Explain.

### Your answers

**Prediction before execution**

_Write your answer here._

**Observed matching count, output excerpt and completion**

_Write your answer here._

**Explanation of the strict condition, lazy evaluation and preview ordering**

_Write your answer here._

## Activity 4 — Count Personal Loan categories

**Purpose:** interpret aggregation: many input records become a small grouped
summary. This activity uses the **full input**, not the age-filtered DataFrame.
The raw category codes are retained; use the source documentation before assigning
meanings to them. The activity describes data, not a trained model or causation.

### Predict

What should the grouped counts sum to? What does sorting by category value change?

### Read the supplied code — do not execute it locally

```python
def count_categories(df) -> None:
    """Activity 4: count raw Personal Loan codes and sort the summary."""
    df.groupBy("Personal Loan").count().orderBy("Personal Loan").show()
```

### Copy to the terminal

```bash
python3 spark_direct_submit.py analyse --activity 4
```

### Observe and reflect

Record each category and its count, their sum, and completion messages.

- Does the sum match the full input count? Explain why this is a useful check.
- Which category is more frequent? What do the documentation and its limitations
  allow you to say about its meaning?
- Why might unequal category frequencies matter in a later classification task?
- Why do grouped results alone not demonstrate execution on multiple workers?

### Your answers

**Prediction before execution**

_Write your answer here._

**Observed category counts, sum and completion**

_Write your answer here._

**Explanation of aggregation, category interpretation and modelling implications**

_Write your answer here._

## Activity 5 — Export and retrieve a bounded preview

**Purpose:** distinguish collecting a small driver-side result from writing a full
distributed dataset, and distinguish remote execution from local retrieval.
The activity recreates the age filter and collects **at most ten records**. An
empty filter result produces a header-only CSV. Do not remove the row limit.

### Predict

Will the file contain all matching records? Where is it written first, and how
will it become available in JupyterLab?

### Read the supplied code — do not execute it locally

```python
def export_preview(result_df, output_dir: Path) -> Path:
    """Collect at most ten filtered rows and write one local preview CSV.

    Args:
        result_df: Spark DataFrame containing Age, Income and Family.
        output_dir: Participant-owned local directory for exported results.

    Returns:
        Path of the CSV written, including its header row.

    The existing filtered_preview.csv is replaced. A remote rerun replaces the
    group's preview; the launcher retrieves it into a new local download directory.
    This differs from Spark's distributed CSV writer, which creates a directory
    with part files. No shared worker directory is modified.
    """
    # Collect the bounded result before opening the output file, so a failed
    # Spark action does not truncate an existing preview.
    rows = result_df.limit(10).collect()
    output_dir.mkdir(exist_ok=True)
    output_file = output_dir / "filtered_preview.csv"

    with output_file.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle)
        writer.writerow(result_df.columns)
        for row in rows:
            writer.writerow(tuple(row))

    return output_file
```

### Copy to the terminal

```bash
python3 spark_direct_submit.py analyse --activity 5
```

### Observe

Check analysis completion and `Remote exit status: 0`, then separately check
`Downloaded preview:`. Open the **printed local download path** in JupyterLab,
not the remote `Saved:` path. For the supplied payload, the download is under:

```text
outputs/remote-<unique-id>/filtered_preview.csv
```

Record its path, header and number of data rows, excluding the header.
Remote reruns replace the shared `outputs/filtered_preview.csv`; each download
gets a fresh local directory. This does not make concurrent submissions safe.
An older preview is not evidence that your current activity succeeded.

### Reflect

- Why is collecting ten records safer than collecting an arbitrarily large result?
- How does this file differ from Spark's distributed CSV output directory?
- Why can remote analysis succeed but local retrieval fail?
- Why do unique local download directories not prevent remote job collisions?

If retrieval fails after a successful exit status, report it; do not automatically
resubmit the job or assume an existing file belongs to your run.

### Your answers

**Prediction before execution**

_Write your answer here._

**Observed remote/local paths, header, row count and completion/retrieval**

_Write your answer here._

**Explanation of bounded collection, retrieval and shared remote output**

_Write your answer here._

## Finish, reflect and report problems

The supplied analysis attempts `spark.stop()` in its existing `finally` block
once a session has been obtained. This does not stop shared containers. Cleanup
can fail or be interrupted. A timeout, SSH closure, closed terminal or Ctrl-C does
**not** prove remote driver termination. The launcher is not a remote process
supervisor. Stop and ask the organiser to inspect an uncertain application before
retrying or allowing another group member to submit.

For missing files/credentials, host-key errors, missing Paramiko, dataset errors,
connection failures, timeouts or cleanup uncertainty, report the command, activity,
failing stage, error text, remote exit status, application ID when available, and
relevant printed paths. **Never include private-key content or credential payloads.**
Do not invent configuration, install software, replace pins, manage shared services,
or repeatedly submit jobs to overcome an error.

### Final reflection

- Trace one activity from the copied terminal command to executor processing and
  driver completion. Which parts happen in JupyterLab and which happen remotely?
- Explain the difference between selection, filtering and aggregation.
- What did automation prepare, and what did you still need to observe and explain?
- What evidence distinguishes completed analysis from completed preview retrieval?

### Completion checklist

- [ ] I used the assigned terminal and authorised launcher commands.
- [ ] My group ran one submission at a time.
- [ ] I recorded predictions, observations and explanations for all five activities.
- [ ] I distinguished independent submissions from a persistent interactive session.
- [ ] I checked each activity's completion and activity 5's separate retrieval.
- [ ] I saved my notebook answers and resolved uncertainties with the organiser.

The original workshop PDF (`docs/spark-DA.pdf`) is conceptual/historical background,
not permission to run old deployment or scaling commands. This is a small teaching
exercise, not a performance benchmark. Step 2 has a separate launch path and
requires its own organiser approval; completing Step 1 does not enable it.

**Optional reference, not an additional required submission:** the unchanged
`python3 spark_direct_submit.py analyse` command runs the existing complete analysis.
Use it only if the organiser requests it; the guided exercise uses the five
numbered activities above.

### Your answers

**End-to-end execution explanation**

_Write your answer here._

**Selection, filtering and aggregation comparison**

_Write your answer here._

**Automation, completion evidence and remaining questions**

_Write your answer here._